In [ ]:
# 一、内部标准化
import sys
sys.path.append('./perturb-seq/script/')
from adtest_mjy import anderson_ksamp_interp  # 导入高精度AD检验函数
import scanpy as sc
import pandas as pd
import numpy as np
from statsmodels.stats.multitest import multipletests
from scipy.sparse import issparse
from itertools import combinations  # 生成两两组合的工具
#（一）收集阴性对照sgRNA细胞

# 1. 读取单细胞数据
adata = sc.read("./perturb-seq/hepg2/GSE264667_hepg2_raw_singlecell_01.h5ad")

# 2. 筛选带有non-阴性对照sgRNA的细胞
# 从gene_transcript列匹配含'non-'的行
neg_ctrl_cells = adata.obs[adata.obs['gene_transcript'].str.contains('non-', na=False)]
# 基于筛选结果更新adata（仅保留阴性对照细胞）
adata_neg_ctrl = adata[neg_ctrl_cells.index, :].copy()

#（二）z标准化

# 计算每个基因的平均UMI计数（按细胞）
gene_mean_umi = adata_neg_ctrl.X.mean(axis=0)
# 若结果是二维数组，用flatten()转为一维
if gene_mean_umi.ndim > 1:
    gene_mean_umi = gene_mean_umi.flatten()
# 筛选平均UMI>1的基因名
high_expr_genes = adata_neg_ctrl.var_names[gene_mean_umi > 1]
# 保留这些基因的表达数据
adata_filtered = adata_neg_ctrl[:, high_expr_genes].copy()

def z_score_normalize(adata):
    """按基因做z标准化（减去均值再除以标准差）"""
    if issparse(adata.X):
        expr = adata.X.toarray()
    else:
        expr = adata.X.copy()
    # 计算均值和标准差
    mean = expr.mean(axis=0)
    std = expr.std(axis=0)
    std[std == 0] = 1e-6  # 避免除以0
    # z标准化
    expr_norm = (expr - mean) / std
    adata.X = expr_norm
    return adata
adata_filtered = z_score_normalize(adata_filtered)

In [10]:
#（三）AD检验
from scipy import sparse
import time
import anndata as ad
from multiprocessing import Pool, Manager
import warnings
warnings.filterwarnings('ignore')

# ad_test_single_gene函数
def ad_test_single_gene(sg1_expr, sg2_expr):
    sg1_expr = sg1_expr[~np.isnan(sg1_expr)]
    sg2_expr = sg2_expr[~np.isnan(sg2_expr)]
    p_val = np.nan
    if len(sg1_expr) > 0 and len(sg2_expr) > 0:
        try:
            # anderson_ksamp_interp函数调用
            p_val = anderson_ksamp_interp([sg1_expr, sg2_expr], midrank=True)
        except Exception as e:
            print(f"检验报错: {e}")
            pass
    return p_val

# 改写ad_test_sg_pair_apply
def ad_test_sg_pair_apply(row, sg_grouped, gene_list):
    sg1, sg2 = row['sg1'], row['sg2']

    # 修复：先判断sgRNA是否在分组中，再提取数据
    sg1_expr_list = []
    sg2_expr_list = []
    for g in gene_list:
        # 提取sg1的基因表达
        if sg1 in sg_grouped.groups:
            sg1_expr = sg_grouped.get_group(sg1)[g].values if g in sg_grouped.obj.columns else np.array([])
        else:
            sg1_expr = np.array([])
        # 提取sg2的基因表达
        if sg2 in sg_grouped.groups:
            sg2_expr = sg_grouped.get_group(sg2)[g].values if g in sg_grouped.obj.columns else np.array([])
        else:
            sg2_expr = np.array([])

        sg1_expr_list.append(sg1_expr)
        sg2_expr_list.append(sg2_expr)

    # 构建基因表达DataFrame
    gene_expr_df = pd.DataFrame({
        'sg1_expr': sg1_expr_list,
        'sg2_expr': sg2_expr_list
    }, index=gene_list)

    # 应用检验函数
    p_series = gene_expr_df.apply(
        lambda x: ad_test_single_gene(x['sg1_expr'], x['sg2_expr']),
        axis=1
    )
    # 封装结果
    result_df = pd.DataFrame({
        'p_value': [p_series.tolist()]
    }, index=[0])
    return result_df

# 改写mp_worker：修正参数传递，避免全局变量依赖
def mp_worker(pair, expr_df, gene_list):
    sg1, sg2 = pair
    # 子进程内独立分组，避免进程间数据冲突
    sg_grouped = expr_df.groupby('sgRNA', observed=False)

    # 构建单行DataFrame供apply调用
    pair_df = pd.DataFrame({'sg1': [sg1], 'sg2': [sg2]})

    # 调用检验函数
    apply_result = pair_df.apply(
        lambda row: ad_test_sg_pair_apply(row, sg_grouped, gene_list),
        axis=1
    )

    # 展开嵌套结果
    apply_result = pd.concat(apply_result.tolist(), ignore_index=True)
    p_series = apply_result['p_value'].iloc[0]

    return f"{sg1};{sg2}", p_series

# 改写load_and_preprocess：修复sg_list生成逻辑
def load_and_preprocess(adata_filtered):
    # 稀疏矩阵转密集
    expr_mat = adata_filtered.X.toarray() if sparse.issparse(adata_filtered.X) else adata_filtered.X
    expr_df = pd.DataFrame(expr_mat, index=adata_filtered.obs.index, columns=adata_filtered.var_names)
    # 修正sgRNA列赋值（确保取到正确的sgRNA字段）
    expr_df['sgRNA'] = adata_filtered.obs['gene_transcript'].values

    # 过滤sgRNA列的空值/无效值
    expr_df = expr_df[expr_df['sgRNA'].notna()]
    expr_df = expr_df[expr_df['sgRNA'] != '']

    # 重新分组，获取有效键
    sg_grouped = expr_df.groupby('sgRNA', observed=False)
    valid_sg_list = list(sg_grouped.groups.keys())  # 从分组键生成有效sgRNA列表

    # 生成sgRNA对
    sg_pairs = list(combinations(valid_sg_list, 2)) if len(valid_sg_list) >=2 else []
    gene_list = adata_filtered.var_names.tolist()

    print(f"有效sgRNA数量: {len(valid_sg_list)}")
    print(f"生成sgRNA对数量: {len(sg_pairs)}")
    print(f"数据形状: {expr_df.shape} | 基因数: {len(gene_list)}")

    # 返回expr_df而非分组字典，供子进程重新分组
    return expr_df, gene_list, sg_pairs, expr_df.shape
# 主执行逻辑
if __name__ == '__main__':
    n_processes = 10

    # 预处理数据
    expr_df, gene_list, sg_pairs, data_shape = load_and_preprocess(adata_filtered)

    # 测试单对sgRNA（绕过多进程，直接调用）
    if sg_pairs:
        test_result = mp_worker(sg_pairs[0], expr_df, gene_list)
        pair_name, p_series = test_result
        print(f"当前处理的pair_name：{pair_name}")
        print(f"单对测试前5个P值: {p_series[:5]}")
    else:
        print("无有效sgRNA对，无法测试")
        exit()

    

有效sgRNA数量: 130
生成sgRNA对数量: 8385
数据形状: (4976, 3027) | 基因数: 3026
当前处理的pair_name：10755_non-targeting_non-targeting_non-targeting;10756_non-targeting_non-targeting_non-targeting
单对测试前5个P值: [0.5995349526910465, 0.035878631883289325, 0.41137851813582493, 0.4267067805554739, 0.10647751996899533]


In [ ]:
#（四）BH校正
import pandas as pd
from statsmodels.stats.multitest import multipletests
# 1. 读取第三步生成的 p 值矩阵
df = pd.read_csv("./sgRNA_pair_test_results.csv")

# 2. 提取所有 p 值列
p_value_columns = df.columns[2:]
p_values_matrix = df[p_value_columns]

# 3. 对所有 p 值执行 Benjamini-Hochberg 校正
# 每一行单独执行BH矫正
fdr_matrix = []
for _, row in p_values_matrix.iterrows():
    _, adj_p, _, _ = multipletests(row.values, method="fdr_bh")
    fdr_matrix.append(adj_p)

# 4. 构建校正后的结果数据框
fdr_df = pd.DataFrame(fdr_matrix, columns=p_value_columns)
# 合并 sg1 和 sg2 列
result_df = pd.concat([df[["sg1", "sg2"]], fdr_df], axis=1)

# 5. 保存结果
result_df.to_csv("sgRNA_pair_test_results_with_fdr.csv", index=False)
print("BH校正完成，结果已保存为 sgRNA_pair_test_results_with_fdr.csv")

BH校正完成，结果已保存为 sgRNA_pair_test_results_with_fdr.csv


In [8]:
test=result_df.iloc[4,2:]
test[test<0.05]

ENSG00000187608    0.045792
ENSG00000179403    0.042485
ENSG00000131910    0.040715
ENSG00000163875    0.028749
ENSG00000143418    0.045519
ENSG00000128699    0.008568
ENSG00000085449    0.018351
ENSG00000012171    0.025573
ENSG00000114315     0.01916
ENSG00000164134    0.043571
ENSG00000204713    0.030915
ENSG00000112062    0.018351
ENSG00000151914    0.015773
ENSG00000029639    0.020572
ENSG00000085662    0.031554
ENSG00000197448    0.029739
ENSG00000165119    0.028104
ENSG00000148180    0.042384
ENSG00000177697    0.033131
ENSG00000168003    0.045792
ENSG00000167770    0.038625
ENSG00000172500    0.010009
ENSG00000110367    0.045519
ENSG00000196935     0.02752
ENSG00000111530    0.045519
ENSG00000100522    0.018351
ENSG00000137876    0.045519
ENSG00000090487    0.045792
ENSG00000008517    0.015773
ENSG00000157045    0.028104
ENSG00000179918    0.045792
ENSG00000051108    0.026896
ENSG00000072778    0.045359
ENSG00000030582    0.015773
ENSG00000108946    0.009613
ENSG00000176890    0

In [ ]:
#（五）计算平均差异基因数
import numpy as np

# 1. 读取上一步的输出结果
result_df = pd.read_csv("./sgRNA_pair_test_results_with_fdr.csv")

# 2. 提取所有潜在对照 sgRNA
potential_controls = pd.unique(result_df[['sg1', 'sg2']].values.ravel())
print(f"共有 {len(potential_controls)} 个潜在对照 sgRNA")

# 3. 定义显著性阈值
fdr_threshold = 0.05

# 4. 计算每一行（每对比较）的差异基因数
# 提取所有 FDR 列（排除 sg1, sg2）
fdr_columns = [col for col in result_df.columns if col not in ['sg1', 'sg2']]
# 对每一行统计 FDR < 阈值的基因数量
result_df['deg_count'] = (result_df[fdr_columns] < fdr_threshold).sum(axis=1)

# 5. 按 sgRNA 分组，计算平均差异基因数
avg_deg_counts = {}
for sgRNA in potential_controls:
    # 找到所有包含该 sgRNA 的行（比较对）
    pairs_with_sgRNA = result_df[(result_df['sg1'] == sgRNA) | (result_df['sg2'] == sgRNA)]
    if pairs_with_sgRNA.empty:
        avg_deg_counts[sgRNA] = 0
        continue
    # 计算这些比较对的平均差异基因数
    avg_deg_counts[sgRNA] = pairs_with_sgRNA['deg_count'].mean()

# 6. 转换为 DataFrame 并排序
avg_deg_df = pd.DataFrame.from_dict(avg_deg_counts, orient='index', columns=['avg_deg_count'])
avg_deg_df = avg_deg_df.sort_values('avg_deg_count')
print("\n每个对照 sgRNA 的平均差异基因数（已按稳定性排序）：")
print(avg_deg_df)

# 7. 保存结果
avg_deg_df.to_csv("control_sgRNA_stability.csv")
print("\n稳定性结果已保存到 control_sgRNA_stability.csv")

共有 130 个潜在对照 sgRNA

每个对照 sgRNA 的平均差异基因数（已按稳定性排序）：
                                                 avg_deg_count
11297_non-targeting_non-targeting_non-targeting       0.193798
10779_non-targeting_non-targeting_non-targeting       0.201550
11068_non-targeting_non-targeting_non-targeting       0.201550
10873_non-targeting_non-targeting_non-targeting       0.263566
11003_non-targeting_non-targeting_non-targeting       0.565891
...                                                        ...
11324_non-targeting_non-targeting_non-targeting     140.038760
10937_non-targeting_non-targeting_non-targeting     140.790698
10994_non-targeting_non-targeting_non-targeting     184.457364
11028_non-targeting_non-targeting_non-targeting     202.348837
10793_non-targeting_non-targeting_non-targeting     394.968992

[130 rows x 1 columns]

稳定性结果已保存到 control_sgRNA_stability.csv


In [23]:
final_controls = avg_deg_df[avg_deg_df['avg_deg_count']<=30].index.tolist()

In [ ]:
#（六）保留低于阈值的潜在对照sRNA
import pandas as pd

# 1.读取csv文件
avg_deg_df = pd.read_csv("./control_sgRNA_stability.csv", index_col=0)

# 2.筛选对照sgRNA
filtered_df = avg_deg_df[avg_deg_df['avg_deg_count'] <= 30]

# 3.保存筛选后的结果到新的文件
filtered_df.to_csv("filter_control_sgRNA_stability.csv", index = True)


In [ ]:
# 二、细胞质量过滤
import pandas as pd
import numpy as np
import scanpy as sc
from scipy import sparse
# 1. 读取筛选后的核心对照sgRNA列表
control_df  = pd.read_csv("./sgRNA_data_scaling/filter_control_sgRNA_stability.csv", index_col = 0)
core_control = control_df.index.tolist()
# 2. 读取原始单细胞数据
adata = sc.read("./perturb-seq/hepg2/GSE264667_hepg2_raw_singlecell_01.h5ad")


In [4]:
adata.obs.columns.tolist()

['gem_group',
 'gene',
 'gene_id',
 'transcript',
 'gene_transcript',
 'sgID_AB',
 'mitopercent',
 'UMI_count',
 'z_gemgroup_UMI']

In [4]:
# 3. 提取核心对照细胞
core_control_mask = adata.obs['gene_transcript'].isin(core_control)
adata_core_controls = adata[core_control_mask].copy()
print(f"原始细胞数：{adata.n_obs}")
print(f"核心对照细胞数：{adata_core_controls.n_obs}")

原始细胞数：145473
核心对照细胞数：3627


In [4]:
adata_core_controls

AnnData object with n_obs × n_vars = 3627 × 9624
    obs: 'gem_group', 'gene', 'gene_id', 'transcript', 'gene_transcript', 'sgID_AB', 'mitopercent', 'UMI_count', 'z_gemgroup_UMI'
    var: 'gene_name', 'chr', 'start', 'end', 'class', 'strand', 'length', 'in_matrix', 'mean', 'std', 'cv', 'fano'

In [5]:
# 4. 按gem_group计算核心对照的平均UMI
core_gem_mean = adata_core_controls.obs.groupby('gem_group')['UMI_count'].mean()
# 5. 计算目标平均UMI（所有gem_group核心对照的均值）
target_mean = core_gem_mean.mean()
# 6. 计算每个gem_group的缩放因子
size_factors = (target_mean / core_gem_mean).to_dict()
print("各gem_group的测序深度缩放因子：")
for gemgroup, sf in  size_factors.items():
    print(f"{gemgroup}: {sf: .4f}")
# 7. 为每个细胞匹配对应的缩放因子
adata.obs['size_factor'] = adata.obs['gem_group'].map(size_factors).fillna(1.0)
# 8. 应用缩放因子校正UMI
adata_corrected = adata.copy()
if sparse.issparse(adata_corrected.X):
    for sf in adata_corrected.obs['size_factor'].unique():
        mask = adata_corrected.obs['size_factor'] == sf
        adata_corrected[mask].X = adata_corrected[mask].X.multiply(sf)
    else:
        adata_corrected.X = adata_corrected.X * adata_corrected.obs['size_factor'].values[:, None]
# 9. 计算校正后的总UMI
adata_corrected.obs['total_umi_corrected'] = adata_corrected.X.sum(axis=1).A1 if sparse.issparse(adata_corrected.X) else adata_corrected.X.sum(axis=1)


各gem_group的测序深度缩放因子：
1:  1.3044
2:  1.1485
3:  1.1689
4:  1.1556
5:  1.0753
6:  1.2196
7:  1.6734
8:  0.9660
9:  1.0180
10:  1.0688
11:  1.3442
12:  1.5157
13:  1.0148
14:  0.5665
15:  1.1058
16:  1.9670
17:  0.6329
18:  1.6246
19:  1.1359
20:  1.2497
21:  1.9675
22:  1.0407
23:  1.1306
24:  0.9584
25:  2.1253
26:  1.1724
27:  1.0796
28:  0.6320
29:  1.3240
30:  1.1275
31:  0.7399
32:  0.4561
33:  1.2763
34:  1.1505
35:  0.4469
36:  1.2633
37:  0.6372
38:  1.4987
39:  0.8375
40:  1.3803
41:  0.4645
42:  1.0838
43:  1.5407
44:  1.3452
45:  0.9650
46:  1.2603
47:  1.0679
48:  1.1576
49:  1.1089
50:  1.0345
51:  0.9667
52:  1.1996
53:  1.1318
54:  1.1596
55:  0.4365
56:  1.2906


In [6]:
# 10. 过滤低质量细胞
# 过滤阈值
MIN_UMI = 3000
MAX_MITO = 0.2

filter_mask = (
    (adata_corrected.obs['total_umi_corrected'] >= MIN_UMI) &
    (adata_corrected.obs['mitopercent'] <= MAX_MITO)
)

adata_filtered = adata_corrected[filter_mask].copy()
print(f"过滤后细胞数：{adata_filtered.n_obs}")

过滤后细胞数：145438


In [7]:
# 三、基因表达矩阵标准化
#（一）UMI计数标准化
# 1. 计算核心对照细胞的中位UMI(校正后的UMI）
# 重新再校正后的数据中提取核心对照细胞
core_control_mask = adata_corrected.obs['gene_transcript'].isin(core_control)
adata_core_controls_corrected = adata_corrected[core_control_mask].copy()
core_control_median_umi = adata_core_controls_corrected.obs['total_umi_corrected'].median()
# 2. 计算每个细胞的UMI缩放因子
# 缩放因子 = 目标中位UMI / 当前细胞总UMI
adata_filtered.obs['umi_scaling_factor'] = (
    core_control_median_umi / adata_filtered.obs['total_umi_corrected']
)
# 3. 应用UMI缩放因子到表达矩阵
adata_umi_normalized = adata_filtered.copy()
if sparse.issparse(adata_umi_normalized.X):
    # 稀疏矩阵处理
    for sf in adata_umi_normalized.obs['umi_scaling_factor'].unique():
        mask = adata_umi_normalized.obs['umi_scaling_factor'] == sf
        adata_umi_normalized[mask].X = adata_umi_normalized[mask].X.multipiy(sf)
else:
    # 密集矩阵处理
    adata_umi_normalized.X = (
        adata_umi_normalized.X * adata_umi_normalized.obs['umi_scaling_factor'].values[:, None]
    )
# 4. 验证：计算标准化后的总UMI，其中位应接近core_control_meidian_umi
adata_umi_normalized.obs['total_umi_normalized'] = (
    adata_umi_normalized.X.sum(axis=1).A1
    if sparse.issparse(adata_umi_normalized.X)
    else adata_umi_normalized.X.sum(axis=1)
)
print(f"标准化后所有细胞的中位UMI：{adata_umi_normalized.obs['total_umi_normalized'].median():.2f}")
print(f"核心对照细胞的中位UMI：{core_control_median_umi:.2f}")

标准化后所有细胞的中位UMI：15834.00
核心对照细胞的中位UMI：15834.00


In [8]:
#（二）相对z标准化
# 1.在adata_umi_normalized中标哪些是核心对照细胞
adata_umi_normalized.obs['is_core_control'] = ( 
    adata_umi_normalized.obs['gene_transcript'].isin(core_control)
)

# 2.按照gem_group和gene分组，计算核心对照细胞的表达均值和标准差
# 核心：矩阵分组广播
# 将gem_group转成分类类型
adata_umi_normalized.obs['gem_group'] = adata_umi_normalized.obs['gem_group'].astype('category')
gem_groups = adata_umi_normalized.obs['gem_group'].cat.categories
# 创建空矩阵
n_cells, n_genes = adata_umi_normalized.shape
control_mean = np.zeros((n_cells, n_genes), dtype=np.float32)
control_std = np.zeros((n_cells, n_genes), dtype=np.float32)
# 开始循环每个gem_group
for gg in gem_groups:
    gg_mask = adata_umi_normalized.obs['gem_group'] == gg
    gg_core_mask = gg_mask & adata_umi_normalized.obs['is_core_control']
    # 若分组没有对照细胞，使用全局均值和标准差
    if gg_core_mask.sum() == 0:
        global_mean = adata_umi_normalized.X.mean(axis=0)
        global_std = adata_umi_normalized.X.std(axis=0)
        if sparse.issparse(adata_umi_normalized.X):
            global_mean = global_mean.A1
            global_std = global_std.A1
        control_mean[gg_mask.values, :] = global_mean
        control_std[gg_mask.values, :] = global_std
        continue

    # 提取该gem_group核心对照的表达矩阵
    adata_gg_core = adata_umi_normalized[gg_core_mask]
    if sparse.issparse(adata_gg_core.X):
        expr_core = adata_gg_core.X.toarray()
    else:
        expr_core = adata_gg_core.X.copy()
    # 按基因计算均值和标准差
    gg_mean = expr_core.mean(axis=0)
    gg_std = expr_core.std(axis=0)
    gg_std[gg_std == 0] = 1e-6
    # 广播到该gem_group所有细胞
    control_mean[gg_mask.values, :] = gg_mean
    control_std[gg_mask.values, :] = gg_std
# 3. 计算相对z分数
# 如果是稀疏矩阵，转成普通矩阵
if sparse.issparse(adata_umi_normalized.X):
    expr = adata_umi_normalized.X.toarray()
else:
    expr = adata_umi_normalized.X.copy()
relative_z = (expr - control_mean) / control_std

# 4. 存入AnnData layers
adata_umi_normalized.layers['relative_z'] = relative_z
print("相对z标准化完成，结果已存入adata.layers['relative_z']")


相对z标准化完成，结果已存入adata.layers['relative_z']
